# Lab 4 · Lập hồ sơ một quận bằng pandas

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 4**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng Bài 4 khám phá toàn thành phố. Trong lab này, bạn lập **hồ sơ chi tiết
của một quận** — Ñuñoa — và so sánh quận này với mặt bằng chung của Santiago.

**Mục tiêu bài lab**

1. Chạy thói quen 5 bước và **rút ra số liệu** từ output, không chỉ thực thi lệnh.
2. Phân biệt `df["cột"]`, `df[["a", "b"]]`, `df[mask]` và `df.loc[mask, "cột"]`; lọc bool nhiều điều kiện; `nsmallest`.
3. Sửa bảng bằng `.loc[hàng, cột]` và **tính lại cột phụ thuộc** sau khi sửa: cột lưu giá trị đã tính, không lưu công thức.
4. Tính tỷ lệ bằng mean-trên-bool và nói rõ **mẫu số**: NaN ảnh hưởng kết quả thế nào.
5. Xuất một bảng hồ sơ ra file CSV, đọc lại và kiểm một bất biến.

Phiên bản đề: `pandas-district-profile-v1` (24/09/2026). Lưu đúng tên **`lab-04.ipynb`**.

## Cách làm và nộp bài

- Bài 1 (khởi động) và các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở:
  được dùng AI, kèm trách nhiệm khai báo theo chính sách AI của môn.
- Ghi dự đoán trước khi chạy; điền thân hàm TODO và phần trả lời Markdown.
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với input riêng, không phụ thuộc biến `df`, `DEMO` hay `TEN_QUAN` ở cell khác.
  `pd` (pandas) và `np` (NumPy) được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên bảng giả lập nhỏ; test ẩn dùng bảng khác cùng hợp đồng
  (index không bắt đầu từ 0, giá thiếu, quận không có trong bảng…).
  Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- **Không sửa DataFrame đầu vào** (không thêm cột, không gán giá trị vào nó); không ghi cứng kết quả.
  Không cần xử lý input ngoài hợp đồng.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.
  Notebook chạy hết khi còn TODO chưa có nghĩa đạt bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1–8, mỗi bài 10 điểm | 80 | Tự động; giảng viên kiểm tra phương pháp |
| Thói quen 5 bước: số liệu đọc từ output | 5 | Giảng viên |
| Dự đoán và giải thích (Bài 1, 5, 6), diễn giải trên dữ liệu thật | 10 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Yêu cầu dùng `.loc`, mask bool, `nsmallest`, `value_counts(normalize=True)` được giảng viên kiểm tra cùng code,
không suy ra chỉ từ kết quả. Bài tự làm ✅ mở là phần mở rộng, không cộng vào 80 điểm tự động.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định notebook mở bằng **12 dòng giả lập** (`DEMO`, giá CLP/đêm) để Run all được cả khi không có mạng.
Không dùng kết quả của 12 dòng này để kết luận về Santiago.

Khi thực hành trên dữ liệu thật, đặt `LISTINGS_PATH = SNAPSHOT_URL` — file `listings.csv` bản **visualisations**,
Santiago, snapshot **2026-06-29**, [Inside Airbnb](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/listings.csv) —
hoặc đường dẫn tới bản sao cùng snapshot do giảng viên cung cấp. Đường dẫn là cấu hình dữ liệu, không phải câu trả lời.

Public checks luôn dùng `DEMO`, nên vẫn chạy được khi chưa có snapshot thật.
Các con số "đối chiếu snapshot" trong đề chỉ để bạn tự kiểm tra khi dùng dữ liệu thật; không dùng chúng làm hằng số trong lời giải.

In [ ]:
import numpy as np
import pandas as pd

# Hành vi pandas 3: sửa qua hai lần ngoặc không đổi bảng gốc (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

SNAPSHOT_URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
                "2026-06-29/visualisations/listings.csv")

# 12 dòng GIẢ LẬP (CLP/đêm) — dùng cho public checks và khi chưa có snapshot thật.
DEMO = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    "name": ["Depto Plaza Ñuñoa", "Pieza cerca metro", "Loft Irarrázaval", "Depto sin precio",
             "Dormitorio simple", "Casa familiar", "Studio Providencia", "Pieza Providencia",
             "Depto Centro", "Hostal Centro", "Pieza Centro", "Hotel boutique"],
    "neighbourhood": ["Ñuñoa"] * 6 + ["Providencia"] * 2 + ["Santiago"] * 4,
    "room_type": ["Entire home/apt", "Private room", "Entire home/apt", "Entire home/apt",
                  "Private room", "Entire home/apt", "Entire home/apt", "Private room",
                  "Entire home/apt", "Shared room", "Private room", "Hotel room"],
    "price": [45000.0, 18000.0, 72000.0, np.nan, 2500.0, 60000.0,
              80000.0, 30000.0, 40000.0, 12000.0, np.nan, 95000.0],
    "number_of_reviews": [12, 3, 0, 1, 25, 7, 4, 0, 30, 2, 5, 9],
})

# Đặt LISTINGS_PATH = SNAPSHOT_URL (hoặc đường dẫn file) để làm trên Santiago thật.
LISTINGS_PATH = None
TEN_QUAN = "Ñuñoa"   # cấu hình: đổi tên quận để lập hồ sơ quận khác

if LISTINGS_PATH is None:
    df = DEMO.copy()
    DATA_LABEL = "GIẢ LẬP — 12 dòng DEMO, không phải Santiago"
else:
    df = pd.read_csv(LISTINGS_PATH)
    DATA_LABEL = f"Dữ liệu người học cung cấp: {LISTINGS_PATH}"

print(DATA_LABEL)
print("shape:", df.shape)

## Bài 1 · Khởi động: chọn cột, chọn hàng và tỷ lệ (~10 phút)

1. Một tên cột → `Series`; một **danh sách** tên cột → `DataFrame` (nhớ 2 lớp ngoặc).
2. `mean` trên cột bool trả về **tỷ lệ** (quy tắc bài 3, áp dụng trên pandas).
3. `[]` hay `.loc`? Ngoặc vuông nhận tên cột hoặc một mask; `.loc` nhận cả hàng lẫn cột.
   `df[mask]` cho các **hàng**, `df.loc[mask, "price"]` cho đúng các **ô** cần lấy.

Dự đoán `type(...)` của từng kết quả trước khi chạy.

### Hợp đồng hàm · 10 điểm tự động

`select_basics(df: pd.DataFrame) -> dict`

Nhận DataFrame có ít nhất các cột `name`, `room_type`, `price` (float, có thể NaN) và ít nhất một dòng.
Trả dict có đúng các khóa:

- `cot_gia`: Series cột `price`;
- `bang_con`: DataFrame gồm đúng hai cột `name`, `price` theo thứ tự đó;
- `ty_le_private`: float, tỷ lệ dòng có `room_type == "Private room"` trên **tổng số dòng** (mean trên bool);
- `nguyen_can`: DataFrame các dòng `room_type == "Entire home/apt"`, đủ mọi cột, lấy bằng `df[mask]`;
- `gia_nguyen_can`: Series `price` của đúng các dòng đó, lấy bằng `df.loc[mask, "price"]`.

Mọi kết quả giữ nguyên index và thứ tự dòng của `df`. Không sửa `df`.
Đối chiếu snapshot: tỷ lệ phòng riêng ≈ **0,184**; **15.004** dòng nguyên căn, trung vị giá nguyên căn **64.900** CLP.

In [ ]:
def select_basics(df: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    before = DEMO.copy()
    result = select_basics(DEMO)
    assert set(result) == {"cot_gia", "bang_con", "ty_le_private", "nguyen_can", "gia_nguyen_can"}
    assert isinstance(result["cot_gia"], pd.Series), "cot_gia phải là Series"
    assert isinstance(result["bang_con"], pd.DataFrame), "bang_con phải là DataFrame — 2 lớp ngoặc"
    assert list(result["bang_con"].columns) == ["name", "price"]
    assert round(result["ty_le_private"], 3) == 0.333
    assert isinstance(result["nguyen_can"], pd.DataFrame) and len(result["nguyen_can"]) == 6
    assert list(result["nguyen_can"].columns) == list(DEMO.columns)
    assert isinstance(result["gia_nguyen_can"], pd.Series)
    assert list(result["gia_nguyen_can"].index) == [0, 2, 3, 5, 6, 8]
    assert result["gia_nguyen_can"].median() == 60000.0
    pd.testing.assert_frame_equal(DEMO, before)

public_check("Q1 · ví dụ công khai", check_q1)

**Dự đoán và giải thích:**

- `type(df["price"])` = …; `type(df[["name", "price"]])` = …
- `df[mask]` khác `df.loc[mask, "price"]` ở chỗ: …
- Trên dữ liệu đang dùng (ghi rõ giả lập hay Santiago): tỷ lệ phòng riêng = …

## Thói quen 5 bước — chạy và đọc kết quả

Tình huống: một nhà đầu tư quan tâm **quận Ñuñoa** và cần bản hồ sơ một trang:
quy mô, mức giá so với thành phố, cơ cấu loại phòng và mức thiếu dữ liệu của quận.

Chạy 5 lệnh dưới, rồi **ghi các con số đọc được** vào ô trả lời (sao chép từ output, không ước đoán).
Phần này do giảng viên đọc (5 điểm), không có hàm chấm tự động.
Đối chiếu snapshot: `(18534, 19)`; `price` thiếu **846** giá trị; **4** loại phòng.

In [ ]:
df.shape

In [ ]:
df.sample(3, random_state=1)[["name", "neighbourhood", "room_type", "price"]]

In [ ]:
df.info()

In [ ]:
df["price"].describe().round(0)

In [ ]:
df["room_type"].value_counts()

**Số liệu đọc từ output** (nguồn dữ liệu: …):

- Số dòng: … · số cột: …
- Số giá thiếu = số dòng − số non-null của `price` trong `info()`: …
- Số loại phòng trong `value_counts()`: …

## Bài 2 · Cắt riêng một quận

Lọc các dòng của quận và tính quận đó chiếm bao nhiêu phần thị trường.
Dùng đúng tên xuất hiện trong output (`"Ñuñoa"` có dấu).

### Hợp đồng hàm · 10 điểm tự động

`filter_district(df: pd.DataFrame, ten_quan: str) -> dict`

Nhận DataFrame có cột `neighbourhood` (chuỗi) và ít nhất một dòng. Trả dict có đúng các khóa:

- `bang`: DataFrame các dòng có `neighbourhood == ten_quan`, đủ mọi cột, giữ index và thứ tự;
- `ty_trong`: float, số dòng của quận chia tổng số dòng của `df`.

Quận không xuất hiện → `bang` rỗng (vẫn đủ cột), `ty_trong = 0.0`. Không sửa `df`.
Đối chiếu snapshot: Ñuñoa có **1.813** chỗ ở ≈ **9,8%** thị trường.

In [ ]:
def filter_district(df: pd.DataFrame, ten_quan: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    result = filter_district(DEMO, "Ñuñoa")
    assert set(result) == {"bang", "ty_trong"}
    assert isinstance(result["bang"], pd.DataFrame) and len(result["bang"]) == 6
    assert list(result["bang"].columns) == list(DEMO.columns)
    assert result["ty_trong"] == 0.5
    empty = filter_district(DEMO, "Las Condes")
    assert len(empty["bang"]) == 0 and empty["ty_trong"] == 0.0

public_check("Q2 · ví dụ công khai", check_q2)

In [ ]:
def thuc_hanh_q2():
    ket_qua = filter_district(df, TEN_QUAN)
    return f"{TEN_QUAN}: {len(ket_qua['bang']):,} chỗ ở = {ket_qua['ty_trong']:.1%} thị trường ({DATA_LABEL})"

preview("Bài 2 trên dữ liệu đang dùng", thuc_hanh_q2)

## Bài 3 · Lọc kết hợp trong quận

Nhà đầu tư hỏi: *"trong Ñuñoa, có bao nhiêu căn **nguyên căn** giá **từ 60.000 CLP trở xuống**?"* —
phân khúc họ muốn cạnh tranh. Nhớ ngoặc quanh từng vế và dùng `&`, không dùng `and`.

### Hợp đồng hàm · 10 điểm tự động

`count_segment(listings: pd.DataFrame, room_type: str = "Entire home/apt", gia_toi_da: float = 60000) -> int`

Nhận bảng một quận (có cột `room_type`, `price` float có thể NaN). Trả **int**: số dòng thỏa
`room_type` bằng tham số **và** `price <= gia_toi_da`. Giá thiếu không được đếm. Bảng rỗng trả 0.
Dùng mask bool, không dùng vòng lặp. Đối chiếu snapshot: **539** căn ở Ñuñoa.

In [ ]:
def count_segment(listings: pd.DataFrame, room_type: str = "Entire home/apt", gia_toi_da: float = 60000) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    result = count_segment(nu_demo)
    assert isinstance(result, (int, np.integer)) and not isinstance(result, bool)
    assert result == 2, "Giá đúng 60.000 cũng được tính (<=); giá thiếu thì không."
    assert count_segment(nu_demo, "Private room", 10000) == 1

public_check("Q3 · ví dụ công khai", check_q3)

In [ ]:
preview("Bài 3 trên dữ liệu đang dùng",
        lambda: count_segment(filter_district(df, TEN_QUAN)["bang"]))

## Bài 4 · Kiểm tra các chỗ ở có giá thấp nhất quận

### Hợp đồng hàm · 10 điểm tự động

`cheapest(listings: pd.DataFrame, n: int = 5) -> pd.DataFrame`

Nhận bảng có cột `name`, `price` (float, có thể NaN) và `1 <= n <=` số dòng **có giá**.
Trả DataFrame **n dòng giá thấp nhất**, chỉ gồm hai cột `name`, `price` theo thứ tự đó,
xếp giá tăng dần, giữ index gốc. Dùng `nsmallest`; đồng giá thì dòng xuất hiện trước đứng trước.
Không sửa input. (Thử `nsmallest` với `n` lớn hơn số dòng để thấy vì sao hợp đồng giới hạn `n`.)
Đối chiếu snapshot: giá thấp nhất Ñuñoa là **2.228** CLP.

In [ ]:
def cheapest(listings: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    result = cheapest(nu_demo, 3)
    assert isinstance(result, pd.DataFrame) and list(result.columns) == ["name", "price"]
    assert result.shape == (3, 2)
    assert list(result.index) == [4, 1, 0] and list(result["price"]) == [2500.0, 18000.0, 45000.0]
    top5 = cheapest(nu_demo, 5)
    assert len(top5) == 5 and top5["price"].notna().all(), "Giá thiếu không được đưa vào danh sách."

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
preview("Bài 4 trên dữ liệu đang dùng",
        lambda: cheapest(filter_district(df, TEN_QUAN)["bang"]))

**Nhận xét:** chỗ ở rẻ nhất là … với giá … CLP/đêm. Mức giá này có hợp lý không? Cần kiểm tra gì thêm? …

*Gợi ý:* trên snapshot, 2.228 CLP/đêm (~60 nghìn đồng) cho một "dormitorio" là mức giá cần được kiểm tra.
Hãy ghi nhận trường hợp này; bài 10 sẽ trình bày quy tắc gắn cờ các giá trị tương tự.

## Bài 5 · Sửa bảng bằng `.loc` và tính lại cột phụ thuộc

Nhà đầu tư muốn xem báo giá **3 đêm** nếu **giảm 10% giá các chỗ ở nguyên căn**.
Làm trên một bản sao để không đụng bảng quận. Chạy ví dụ dưới và dự đoán trước:
sau khi sửa `price`, cột `tien_3_dem` có tự đổi theo giá mới không?

### Hợp đồng hàm · 10 điểm tự động

`discount_quote(listings: pd.DataFrame, room_type: str = "Entire home/apt", giam: float = 0.10) -> pd.DataFrame`

Nhận bảng có cột `id`, `room_type`, `price` (float, có thể NaN); `0 <= giam < 1`.
Trả **bảng mới** gồm đúng các cột `id`, `room_type`, `price`, `tien_3_dem`, giữ index và thứ tự:
`price` của các dòng có `room_type` bằng tham số được nhân `(1 - giam)` **bằng `.loc[mask, "price"]`**,
các dòng khác giữ giá; `tien_3_dem = price * 3` tính từ **giá sau khi giảm**. Giá NaN vẫn là NaN.
Không sửa `listings` (làm trên `.copy()`), không dùng hai lần ngoặc `bang["price"][mask] = ...`.
Đối chiếu snapshot Ñuñoa: **1.457** dòng nguyên căn; trung vị giá sau giảm **55.050,3**;
trung vị `tien_3_dem` **182.943** nếu quên tính lại, **165.150,9** sau khi tính lại.

In [ ]:
bao_gia = DEMO[["id", "room_type", "price"]].copy()
bao_gia["tien_3_dem"] = bao_gia["price"] * 3
mask_demo = bao_gia["room_type"] == "Entire home/apt"
bao_gia.loc[mask_demo, "price"] = bao_gia.loc[mask_demo, "price"] * 0.9
# tien_3_dem vẫn là con số tính từ giá CŨ: cột lưu giá trị, không lưu công thức
print(bao_gia.head(4))

In [ ]:
def discount_quote(listings: pd.DataFrame, room_type: str = "Entire home/apt", giam: float = 0.10) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    before = nu_demo.copy()
    result = discount_quote(nu_demo)
    assert isinstance(result, pd.DataFrame)
    assert list(result.columns) == ["id", "room_type", "price", "tien_3_dem"]
    assert list(result.index) == list(nu_demo.index)
    np.testing.assert_allclose(result["price"], [40500, 18000, 64800, np.nan, 2500, 54000])
    np.testing.assert_allclose(result["tien_3_dem"], [121500, 54000, 194400, np.nan, 7500, 162000],
                               err_msg="tien_3_dem phải tính từ giá SAU khi giảm")
    pd.testing.assert_frame_equal(nu_demo, before, obj="bảng đầu vào (không được sửa)")

public_check("Q5 · ví dụ công khai", check_q5)

**Dự đoán và giải thích:** sau khi sửa `price`, `tien_3_dem` … vì …

Nếu viết `bao_gia["price"][mask] = ...` (hai lần ngoặc) thì … vì …

## Bài 6 · So sánh với thành phố và ảnh hưởng của NaN

So từng chỗ ở của quận với **trung vị toàn thành phố** `df["price"].median()`
(snapshot: 59.000 CLP). Tính tỷ lệ chỗ ở có giá **thấp hơn** ngưỡng theo 2 mẫu số:

- cách 1 (thô): mean trực tiếp trên `prices < nguong`;
- cách 2 (sạch): chỉ tính trên các hàng **có giá**.

### Hợp đồng hàm · 10 điểm tự động

`share_below(prices: pd.Series, nguong: float) -> dict`

Nhận Series giá float (có thể NaN, có ít nhất một giá hợp lệ) và ngưỡng hữu hạn. Trả dict có đúng các khóa:
`ty_le_tho` (float, mẫu số = mọi dòng), `ty_le_sach` (float, mẫu số = các dòng có giá),
`so_thieu` (int, số giá NaN). Không làm tròn, không sửa input.
Đối chiếu snapshot Ñuñoa: thô **0,444**, sạch **0,467**.

In [ ]:
def share_below(prices: pd.Series, nguong: float) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    med_demo = DEMO["price"].median()
    assert med_demo == 42500.0
    result = share_below(nu_demo["price"], med_demo)
    assert set(result) == {"ty_le_tho", "ty_le_sach", "so_thieu"}
    assert round(result["ty_le_tho"], 3) == 0.333, "Cách thô: NaN < ngưỡng cho False nhưng vẫn ở mẫu số"
    assert result["ty_le_sach"] == 0.4 and result["so_thieu"] == 1

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
preview("Bài 6 trên dữ liệu đang dùng",
        lambda: share_below(filter_district(df, TEN_QUAN)["bang"]["price"], df["price"].median()))

**Giải thích:** hai tỷ lệ lệch nhau vì … Khi báo cáo, bạn chọn cách nào và nói rõ mẫu số thế nào? …

## Bài 7 · Cơ cấu loại phòng và mức thiếu — hoàn thiện hồ sơ

### Hợp đồng hàm · 10 điểm tự động

`room_mix(listings: pd.DataFrame) -> dict`

Nhận bảng có ít nhất một dòng, cột `room_type` (chuỗi, không thiếu) và `price` (float, có thể NaN).
Trả dict có đúng các khóa:

- `co_cau`: Series `value_counts(normalize=True)` của `room_type` — index là loại phòng, giá trị là tỷ lệ 0–1,
  chỉ gồm các loại có mặt, không làm tròn;
- `thieu_pct`: float, phần trăm dòng thiếu giá (thang 0–100), **làm tròn 2 chữ số**.

Đối chiếu snapshot Ñuñoa: nguyên căn **0,804**; thiếu giá **4,96%** (toàn thành phố 4,56%).

In [ ]:
def room_mix(listings: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q7():
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    result = room_mix(nu_demo)
    assert set(result) == {"co_cau", "thieu_pct"}
    assert isinstance(result["co_cau"], pd.Series)
    assert round(result["co_cau"]["Entire home/apt"], 3) == 0.667
    assert abs(result["co_cau"].sum() - 1) < 1e-12
    assert result["thieu_pct"] == 16.67

public_check("Q7 · ví dụ công khai", check_q7)

In [ ]:
preview("Bài 7 trên dữ liệu đang dùng", lambda: room_mix(filter_district(df, TEN_QUAN)["bang"]))

## Bài 8 · Xuất hồ sơ ra file và đọc lại

### Hợp đồng hàm · 10 điểm tự động

`export_profile(listings: pd.DataFrame, ten_quan: str, path) -> pd.DataFrame`

Nhận bảng một quận (có cột `room_type`, `price`; ít nhất một giá hợp lệ), tên quận và đường dẫn file CSV
(thư mục cha đã tồn tại, cho phép ghi đè). Tạo DataFrame **một dòng** với đúng các cột theo thứ tự:

| Cột | Giá trị |
|---|---|
| `quan` | `ten_quan` |
| `so_cho_o` | số dòng của bảng |
| `gia_trung_vi` | trung vị `price` (bỏ NaN) |
| `ty_le_nguyen_can` | tỷ lệ dòng `Entire home/apt`, làm tròn 3 chữ số |
| `ty_le_thieu_gia_pct` | phần trăm dòng thiếu giá, làm tròn 2 chữ số |

Ghi bằng `to_csv(path, index=False)` (UTF-8), rồi **đọc lại** bằng `pd.read_csv(path)` và trả bảng đọc lại.
Không sửa input. Đối chiếu snapshot Ñuñoa: 1.813 chỗ ở, trung vị **60.981** CLP (nhỉnh hơn thành phố 3%).

In [ ]:
def export_profile(listings: pd.DataFrame, ten_quan: str, path) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q8():
    import tempfile
    from pathlib import Path
    nu_demo = DEMO[DEMO["neighbourhood"] == "Ñuñoa"]
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "ho_so.csv"
        doc_lai = export_profile(nu_demo, "Ñuñoa", path)
        assert path.exists(), "Cần thực sự ghi file."
        header = path.read_text(encoding="utf-8").splitlines()[0]
    assert header == "quan,so_cho_o,gia_trung_vi,ty_le_nguyen_can,ty_le_thieu_gia_pct", "Kiểm tra index=False"
    assert isinstance(doc_lai, pd.DataFrame) and doc_lai.shape == (1, 5)
    assert doc_lai.loc[0, "quan"] == "Ñuñoa" and doc_lai.loc[0, "so_cho_o"] == 6
    assert doc_lai.loc[0, "gia_trung_vi"] == 45000.0
    assert doc_lai.loc[0, "ty_le_nguyen_can"] == 0.667 and doc_lai.loc[0, "ty_le_thieu_gia_pct"] == 16.67

public_check("Q8 · ví dụ công khai", check_q8)

In [ ]:
def thuc_hanh_q8():
    ten_file = "ho_so_" + TEN_QUAN.lower().replace(" ", "_") + ".csv"
    doc_lai = export_profile(filter_district(df, TEN_QUAN)["bang"], TEN_QUAN, ten_file)
    # Bất biến: đọc lại vẫn đúng số chỗ ở và trung vị của bảng quận.
    assert doc_lai.loc[0, "so_cho_o"] == len(filter_district(df, TEN_QUAN)["bang"])
    return doc_lai

preview("Bài 8 trên dữ liệu đang dùng", thuc_hanh_q8)

## Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà) — không cộng vào 80 điểm tự động

Bạn được dùng AI theo quy trình 5 bước; hãy ghi lại prompt chính và cách kiểm chứng.

### Tự làm 1 · Hồ sơ quận thứ hai

Chạy lại Bài 2–8 cho một quận bạn tự chọn (gợi ý: Providencia hoặc Estación Central) và
viết 3 câu so sánh nó với Ñuñoa. Code đã được viết sao cho **chỉ cần đổi một biến `TEN_QUAN`**
là chạy được cho quận bất kỳ; biến này đóng vai trò cấu hình.

### Tự làm 2 · Quận có giá thấp và nhiều lựa chọn

Dùng `df.groupby("neighbourhood")["price"]` với `.agg(["median", "size"])` (xem trước
bài 5): tìm quận có **≥ 300 chỗ ở** và giá trung vị **thấp nhất**. Kiểm chứng lại con số
bằng cách lọc thủ công quận đó và gọi `.median()`.

### Tự làm 3 · AI là đề bài — vì sao bảng không đổi?

Một lời giải do AI sinh ra cho Bài 5:

```python
bao_gia = nu[["id", "room_type", "price"]].copy()
bao_gia["price"][bao_gia["room_type"] == "Entire home/apt"] *= 0.9
```

Chạy thử, đọc thông báo pandas đưa ra, giải thích bằng 2–3 câu vì sao giá không đổi, rồi viết lại đúng bằng `.loc`.

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** (ghi quận đã chọn, 3 câu so sánh, kết quả Tự làm 2 và giải thích Tự làm 3)

…

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu phép tính tay, số dòng hoặc ví dụ đối chiếu đã dùng.

---

## Tóm tắt bài lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Thói quen 5 bước + đọc số liệu từ output | kiểm tra dataset mới và code do AI tạo |
| `df[mask]` vs `df.loc[mask, cột]`, lọc nhiều điều kiện, `nsmallest` | mọi bước lọc/chọn từ Bài 5 trở đi |
| Sửa bảng bằng `.loc`, tính lại cột phụ thuộc, bẫy hai lần ngoặc | làm sạch dữ liệu (Bài 10), pipeline BTL |
| NaN làm lệch mean-trên-bool (0,444 vs 0,467) | làm sạch dữ liệu (bài 10) |
| Hồ sơ quận → CSV, code đổi 1 biến chạy quận khác | xây dựng phân tích có thể tái sử dụng |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài giảng tiếp theo:** index, `groupby` và ghép bảng để tổng hợp nhiều quận.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-04.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_04.ipynb`).
2. Hoàn thành hàm TODO, số liệu thói quen 5 bước, các giải thích và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
   File `ho_so_*.csv` do Bài 8 tạo ra không cần commit.
4. Tạo tag `submit/lab-04/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-04` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()